# NIH ChestX-ray14 — BioViL-T baseline on the official split

Trains one multi-label classifier (14 findings) on the **official NIH split**, the same split as the cascade study
(`evaluation_protocol.yaml`, split id `NIH_OFFICIAL_PV`):
* **test** = every image in `test_list.txt` (25,596 images)
* **val** = random 15 % of the *patients* in `train_val_list.txt` (seed 42) — 12,608 images
* **train** = the remaining train_val patients — 73,916 images

The split is rebuilt with the same code as `2-staged_official_split.ipynb`. The notebook **stops** if the
split hash is not `b9f2fbc9…`, so the test set is exactly the one M08/M09 were scored on.

**Model:** BioViL-T image encoder (Microsoft, `hi-ml-multimodal`) — a hybrid ResNet-50 + Transformer, pretrained
with image–text contrastive learning on MIMIC-CXR (no NIH images, so no test leakage). Used in single-image mode.
The package's own preprocessing: resize 512 → center-crop 448, grayscale → 3 channels, no ImageNet normalization.
New 14-way head on the 512-d pooled feature. Two phases: frozen-backbone warm-up → full fine-tune.
Checkpoint = best val macro AUROC. Thresholds are tuned on val and applied to test.
Microsoft's model card marks BioViL-T as research use only.

**Expected run time (T4):** about 5–9 h for the full run (448×448 input). Warm-up is 2 epochs (was 4) to fit
Kaggle's 12-hour limit; with 73,916 images one epoch is ~6,000 head updates, plenty for a linear head.
A time guard (`TRAIN_BUDGET_H`) stops training early if needed, so the notebook always finishes and saves.

**How to run on Kaggle**
1. *Add Input* → search **`nih-chest-xrays/data`** (NIH Chest X-rays, 112,120 images, ~45 GB). It already contains `train_val_list.txt` and `test_list.txt`.
2. *Settings* → Accelerator **GPU T4** (one GPU is used), **Internet ON** (needed for `pip install` and the pretrained weights).
3. Set `DEBUG = True` in the Config cell → *Run All* → check it finishes without errors (~15–20 min, most of it listing images and installing). Numbers from a DEBUG run mean nothing.
4. Set `DEBUG = False` → *Save Version* → **Save & Run All (Commit)**. It runs in the background; you can close the browser.
5. When it finishes: open the version → *Output* tab → download `results.json`, `split_NIH_OFFICIAL_PV.csv`, `figures/` and the `NIH_CXR14__NIH_OFFICIAL_PV__…__s42/` folder.

In [ ]:
# ── Install / import ────────────────────────────────────────────────────────
!pip install -q hi-ml-multimodal

import os, glob, json, time, random, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image, UnidentifiedImageError

from health_multimodal.image.utils import get_image_inference, ImageModelType
from health_multimodal.image.data.io import load_image
from health_multimodal.image.data.transforms import ExpandChannels
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# ── Config ───────────────────────────────────────────────────────────────────
DEBUG = False        # True = quick smoke test (2,000 train / 500 val / 500 test images, 1+1 epochs). Never report DEBUG numbers.

INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./outputs")
CKPT_DIR = OUTPUT_DIR / "checkpoints"
FIGURES_DIR = OUTPUT_DIR / "figures"
for d in [OUTPUT_DIR, CKPT_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Official split - identical to 2-staged_official_split.ipynb (evaluation_protocol.yaml, NIH_OFFICIAL_PV) ──
# test = every image in test_list.txt; val = random 15 % of the train_val PATIENTS (seed 42); train = the rest.
DATASET_ID, SPLIT_ID = "NIH_CXR14", "NIH_OFFICIAL_PV"
VAL_FRACTION, SPLIT_SEED = 0.15, 42
EXPECTED_COUNTS = {"train_val": 86524, "test": 25596, "total": 112120}
EXPECTED_LIST_SHA256 = {                                   # SHA-256 of the sorted, LF-joined image ids
    "train_val_list.txt": "61fbe896321c1c1c8b75f3e4f3a08e4fef6486d95ef8a667c31d4d60dca6cb81",
    "test_list.txt":      "38ca5ef7f756092946f57c1a59faca882ed589a1ab1f72590b45dc06c6d5e1cc"}
EXPECTED_SPLIT_SHA256 = "b9f2fbc9aa061dafa76994494b92779651dc3ddd2ed3e7b82c70bbf5cf7071dd"   # split of the seed-42 cascade runs
EXPECTED_SIZES = {"train": 73916, "val": 12608, "test": 25596}

# 14 NIH CXR-14 pathology labels ("No Finding" is implicit: all-zero row) - same order as the protocol
DISEASE_LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia",
]
NUM_CLASSES = len(DISEASE_LABELS)

BACKBONE = "biovil_t"   # health_multimodal - MIMIC-CXR contrastive-pretrained hybrid ResNet50+Transformer
MODEL_ID = "BIOVIL_T_448"

# BioViL-T's own preprocessing pipeline (verified against the library source):
# resize -> center-crop -> grayscale expanded to 3 channels. No ImageNet
# mean/std normalization - the model was not trained with it.
RESIZE_SIZE = 512
CROP_SIZE = 448
IMAGE_SIZE = CROP_SIZE           # 448 - BioViL-T's native crop resolution
DROPOUT = 0.4
TEMPERATURE_INIT = 1.5

BATCH_SIZE = 12           # 448x448 + a heavier hybrid ResNet+Transformer backbone - watch GPU memory
LR_WARMUP = 1e-3
LR_FINETUNE = 2e-5   # same conservative LR that fixed the DenseNet-121 divergence bug
EPOCHS_WARMUP = 2    # 2 (was 4): one full-data epoch is ~6,000 head updates; saves ~1.5 h for fine-tuning
EPOCHS_FINETUNE = 6

TRAIN_BUDGET_H = 9.5   # stop training before this many hours; leaves time for evaluation inside Kaggle's 12 h limit
if DEBUG:
    EPOCHS_WARMUP, EPOCHS_FINETUNE = 1, 1

RUN_DIR = OUTPUT_DIR / f"{DATASET_ID}{'_debug' if DEBUG else ''}__{SPLIT_ID}__{MODEL_ID}__s{SEED}"

# Colors (validated categorical / sequential palette — colorblind-safe)
C_BLUE, C_ORANGE, C_AQUA = "#2a78d6", "#eb6834", "#1baf7a"

## 1. Data Loading

Loads the **full** NIH ChestX-ray14 metadata (`Data_Entry_2017.csv`, 112,120 images) and the two
official list files (`train_val_list.txt`, `test_list.txt`, shipped with the Kaggle dataset).

What changed from the sample-dataset version:
* **No rows are dropped.** The old age filter and corrupt-image filter are gone: the official test set
  must contain every one of its 25,596 images, and the split hash is computed over all 112,120 rows.
* The list files are checked against the protocol's SHA-256 hashes, so a wrong or edited list stops the notebook.
* Labels are exact matches on the `|`-separated `Finding Labels` string.

In [ ]:
def find_file(fname, start=INPUT_DIR):
    """Search /kaggle/input for a file, skipping the 12 big image folders."""
    for root, dirs, files in os.walk(start):
        dirs[:] = [d for d in dirs if not d.startswith("images")]
        if fname in files:
            return Path(root) / fname
    return None

CSV_PATH = find_file("Data_Entry_2017.csv")
assert CSV_PATH is not None, "Data_Entry_2017.csv not found - attach the Kaggle dataset 'nih-chest-xrays/data'"
DATA_ROOT = CSV_PATH.parent
print(f"Dataset root: {DATA_ROOT}")

# filename -> full Path for every PNG (full dataset layout: images_001 ... images_012 / images / *.png)
png = glob.glob(str(DATA_ROOT / "images_*" / "images" / "*.png")) or [str(p) for p in DATA_ROOT.rglob("*.png")]
image_index = {os.path.basename(p): Path(p) for p in png}
print(f"Images found on disk: {len(image_index):,}")

df = pd.read_csv(CSV_PATH)
assert len(df) == EXPECTED_COUNTS["total"], f"expected {EXPECTED_COUNTS['total']:,} rows, got {len(df):,} - is this the FULL dataset?"
assert df["Image Index"].isin(image_index).all(), "some images listed in Data_Entry_2017.csv are missing on disk"

# Multi-hot encode the 14 disease labels (exact match on the '|'-separated string)
finding_lists = df["Finding Labels"].str.split("|")
for label in DISEASE_LABELS:
    df[label] = finding_lists.apply(lambda L, c=label: c in L).astype(int)
df["No_Finding"] = (df[DISEASE_LABELS].sum(axis=1) == 0).astype(int)
df = df.sort_values("Image Index").reset_index(drop=True)      # canonical order - the split hash depends on it

# ── Official list files (QC-01) ─────────────────────────────────────────────
def read_list(fname):
    p = DATA_ROOT / fname
    p = p if p.exists() else find_file(fname)
    assert p is not None, f"{fname} not found - it ships with nih-chest-xrays/data; otherwise upload your copy as a Kaggle dataset"
    return open(p).read().split()

tv_ids, te_ids = read_list("train_val_list.txt"), read_list("test_list.txt")
list_sha = {"train_val_list.txt": hashlib.sha256("\n".join(sorted(tv_ids)).encode()).hexdigest(),
            "test_list.txt":      hashlib.sha256("\n".join(sorted(te_ids)).encode()).hexdigest()}
assert (len(tv_ids), len(te_ids)) == (EXPECTED_COUNTS["train_val"], EXPECTED_COUNTS["test"]), "QC-01: list counts"
assert list_sha == EXPECTED_LIST_SHA256, "QC-01: list files differ from the official lists"
assert not set(tv_ids) & set(te_ids) and set(tv_ids) | set(te_ids) == set(df["Image Index"]), "QC-01: overlap/coverage"
print("QC-01 passed: official list files, counts and coverage OK")

df.head()

In [ ]:
print("Per-label positive counts (all 112,120 images):\n")
counts = df[DISEASE_LABELS + ["No_Finding"]].sum().sort_values(ascending=False)
print(counts.to_string())

## 2. Exploratory Data Analysis

In [ ]:
# Class frequency — a single-series magnitude comparison, so one hue (sequential blue)
counts_sorted = df[DISEASE_LABELS].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(counts_sorted.index, counts_sorted.values, color=C_BLUE, height=0.65)
ax.set_xlabel("Number of positive images")
ax.set_title("Label frequency (all 112,120 images)")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "label_frequency.png", dpi=140)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Findings-per-image histogram (single series -> single hue)
n_findings = df[DISEASE_LABELS].sum(axis=1)
axes[0].hist(n_findings, bins=range(0, int(n_findings.max()) + 2), color=C_BLUE,
             align="left", rwidth=0.7)
axes[0].set_xlabel("Number of findings per image")
axes[0].set_ylabel("Image count")
axes[0].set_title("Findings per image")
axes[0].spines[["top", "right"]].set_visible(False)

# Gender split (2 categories -> categorical slots 1 & 2, validated pair)
gender_counts = df["Patient Gender"].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values, color=[C_BLUE, C_ORANGE], width=0.5)
axes[1].set_title("Patient gender")
axes[1].set_ylabel("Image count")
axes[1].spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "eda_overview.png", dpi=140)
plt.show()


In [ ]:
# Sample images with their labels
sample_rows = df.sample(n=min(8, len(df)), random_state=SEED)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    img = Image.open(image_index[row["Image Index"]]).convert("L")
    ax.imshow(img, cmap="gray")
    labels = [l for l in DISEASE_LABELS if row[l] == 1] or ["No Finding"]
    ax.set_title(", ".join(labels), fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "sample_images.png", dpi=140)
plt.show()


## 3. Official Train / Val / Test Split

* **test** = every image in `test_list.txt` (25,596 images, 2,797 patients).
* **val** = random 15 % of the *patients* in `train_val_list.txt` (`GroupShuffleSplit`, seed 42).
* **train** = the remaining train_val patients.

This is the **same code** that built the split for the cascade study (`2-staged_official_split.ipynb`).
The notebook stops if the split hash is not `b9f2fbc9…`. That guarantees the test set and the validation
set are exactly the ones M08/M09 were evaluated on, so the numbers are comparable.
No patient appears in more than one partition.

In [ ]:
df["split"] = np.where(df["Image Index"].isin(set(te_ids)), "test", "trainval")
tv_frame = df[df.split == "trainval"]
_, va = next(GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SPLIT_SEED)
             .split(tv_frame, groups=tv_frame["Patient ID"]))
df.loc[tv_frame.index, "split"] = "train"
df.loc[tv_frame.index[va], "split"] = "val"

split_text = (df[["Image Index", "Patient ID", "split"]]
              .rename(columns={"Image Index": "image_index", "Patient ID": "patient_id", "split": "partition"})
              .sort_values("image_index").to_csv(index=False, lineterminator="\n"))
SPLIT_SHA256 = hashlib.sha256(split_text.encode("utf-8")).hexdigest()
assert SPLIT_SHA256 == EXPECTED_SPLIT_SHA256, \
    f"split hash {SPLIT_SHA256[:12]} != {EXPECTED_SPLIT_SHA256[:12]}: a library version changed the validation carve-out"
with open(OUTPUT_DIR / f"split_{SPLIT_ID}.csv", "w", encoding="utf-8", newline="\n") as f:
    f.write(split_text)

pats = {s: set(df.loc[df.split == s, "Patient ID"]) for s in ("train", "val", "test")}
assert not (pats["train"] & pats["val"] or pats["train"] & pats["test"] or pats["val"] & pats["test"]), "QC-02: patient leakage"

train_df, val_df, test_df = (df[df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
assert {"train": len(train_df), "val": len(val_df), "test": len(test_df)} == EXPECTED_SIZES
print(f"QC-02/03 passed: split {SPLIT_ID} ({SPLIT_SHA256[:12]}), no patient leakage")

if DEBUG:   # smoke test only - never report these numbers
    train_df, val_df, test_df = (d.sample(n=n, random_state=SEED).sort_values("Image Index").reset_index(drop=True)
                                 for d, n in ((train_df, 2000), (val_df, 500), (test_df, 500)))
    print("[DEBUG] subsampled to 2,000 / 500 / 500 images")

print(f"Patients   — train: {train_df['Patient ID'].nunique():,}  val: {val_df['Patient ID'].nunique():,}  test: {test_df['Patient ID'].nunique():,}")
print(f"Images     — train: {len(train_df):,}  val: {len(val_df):,}  test: {len(test_df):,}")

# Per-class pos_weight for BCEWithLogitsLoss (handles class imbalance).
# Raw weights hit 490-560x for Hernia — capping with sqrt() keeps rare
# classes up-weighted without the extreme swing that causes over-prediction
# at inference (this was the main driver of the earlier low-precision runs).
pos = train_df[DISEASE_LABELS].sum(axis=0).values.astype(np.float32)
neg = len(train_df) - pos
pos_weight_raw = torch.tensor(neg / np.maximum(pos, 1.0), dtype=torch.float32)
pos_weight = torch.sqrt(pos_weight_raw)
print("\npos_weight per class (raw -> capped):")
for l, w_raw, w in zip(DISEASE_LABELS, pos_weight_raw.tolist(), pos_weight.tolist()):
    print(f"  {l:<20s} {w_raw:8.2f}  ->  {w:6.2f}")

## 4. Dataset & Transforms

Uses BioViL-T's own preprocessing (resize -> center-crop -> channel-expand),
matched exactly to the library's source rather than reinvented, since this
model was trained without ImageNet-style mean/std normalization - adding it
would push inputs outside the range the backbone was pretrained on. Still
**no horizontal flip** (laterality), mild rotation only for training.


In [ ]:
train_transform = T.Compose([
    T.Resize(RESIZE_SIZE),
    T.RandomRotation(degrees=10),                      # mild only — no h-flip (laterality)
    T.CenterCrop(CROP_SIZE),
    T.ToTensor(),
    ExpandChannels(),                                   # 1-channel -> 3-channel (BioViL-T's own transform)
])

eval_transform = T.Compose([
    T.Resize(RESIZE_SIZE),
    T.CenterCrop(CROP_SIZE),
    T.ToTensor(),
    ExpandChannels(),
])


class BioViLTChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_index, transform):
        self.df = dataframe.reset_index(drop=True)
        self.image_index = image_index          # filename -> full Path
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # load_image (from health_multimodal) handles PNG -> 8-bit grayscale
        # remap the same way BioViL-T's own pretraining pipeline did.
        image = load_image(self.image_index[row["Image Index"]])
        image = self.transform(image)
        labels = torch.tensor(row[DISEASE_LABELS].values.astype(np.float32))
        return image, labels


train_ds = BioViLTChestXrayDataset(train_df, image_index, train_transform)
val_ds = BioViLTChestXrayDataset(val_df, image_index, eval_transform)
test_ds = BioViLTChestXrayDataset(test_df, image_index, eval_transform)

# Weighted sampler: up-weight images with fewer co-occurring findings (unchanged policy)
label_sums = train_ds.df[DISEASE_LABELS].sum(axis=1).values
sample_weights = np.where(label_sums > 0, 1.0 / np.maximum(label_sums, 1e-8), 1.0)
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float64),
    num_samples=len(train_ds), replacement=True,
    generator=torch.Generator().manual_seed(SEED),
)

NUM_WORKERS = 4
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

images, labels = next(iter(train_loader))
print(f"Batch — images: {images.shape}  labels: {labels.shape}")

## 5. Model — BioViL-T + Temperature Scaling

Backbone is BioViL-T's image encoder, pretrained via image-text contrastive
learning on MIMIC-CXR (a different hospital's chest X-rays + their radiology
reports). We use the pre-projection `img_embedding` (512-dim, before the
CLIP-style projection head) as the feature vector, with the project's own
14-way classifier head on top, plus the same learned-temperature calibration
layer as the other three models.


In [ ]:
class TemperatureScaler(nn.Module):
    """Post-hoc calibration: logits / T. T is tuned after training, on val data."""

    def __init__(self, init_temp: float = TEMPERATURE_INIT):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.log(torch.tensor(init_temp, dtype=torch.float32)))

    @property
    def temperature(self):
        return self.log_temperature.exp()

    def forward(self, logits):
        return logits / self.temperature


class BioViLTClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, dropout=DROPOUT):
        super().__init__()
        engine = get_image_inference(image_model_type=ImageModelType.BIOVIL_T)
        self.backbone = engine.model            # hybrid ResNet50 + Transformer, feature_size=512
        self.feature_dim = self.backbone.feature_size
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(self.feature_dim, num_classes),
        )
        self.temp_scaler = TemperatureScaler()

    def forward_logits(self, x):
        feats = self.backbone(x).img_embedding   # (B, 512) pre-projection pooled feature
        return self.classifier(feats)

    def forward(self, x):
        return self.temp_scaler(self.forward_logits(x))


def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = ("classifier" in name) or ("temp_scaler" in name)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"[freeze_backbone] trainable: {trainable:,} / {total:,}")


def unfreeze_backbone(model):
    for p in model.parameters():
        p.requires_grad = True
    print(f"[unfreeze_backbone] trainable: {sum(p.numel() for p in model.parameters()):,}")


model = BioViLTClassifier().to(device)
print(f"Feature dim: {model.feature_dim}   Initial temperature: {model.temp_scaler.temperature.item():.3f}")


## 6. Training — Two-Phase (frozen-backbone warm-up → full fine-tune)

**Phase 1 (warm-up):** freeze the backbone, train only the classifier head — the head learns without
disturbing the chest X-ray pretrained features.
**Phase 2 (fine-tune):** unfreeze everything, train end-to-end at a lower learning rate with `ReduceLROnPlateau`.

Loss is class-weighted `BCEWithLogitsLoss` (multi-label). The checkpoint is chosen by **val macro AUROC**.
Training uses fp16 autocast on the GPU (about 2x faster on a T4). A time guard stops training before
`TRAIN_BUDGET_H` hours, so the notebook always finishes inside Kaggle's 12-hour limit and saves its outputs.

In [ ]:
USE_AMP = device.type == "cuda"          # fp16 autocast on the T4
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(images)
        loss = criterion(logits.float(), labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
    return running_loss / max(len(loader), 1)


@torch.no_grad()
def validate(model, loader, criterion, device, num_classes=NUM_CLASSES):
    """Returns (val_loss, val_macro_auroc). We checkpoint on AUROC, not loss —
    loss can keep rising from probability-scale drift even while the model's
    ability to RANK sick-vs-healthy (what AUROC measures) keeps improving."""
    model.eval()
    running_loss = 0.0
    all_probs, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model(images).float()
        running_loss += criterion(logits, labels).item()
        all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    probs = np.concatenate(all_probs)
    labels_np = np.concatenate(all_labels)
    aurocs = [roc_auc_score(labels_np[:, i], probs[:, i])
              for i in range(num_classes) if len(np.unique(labels_np[:, i])) > 1]
    macro_auroc = float(np.mean(aurocs)) if aurocs else float("nan")
    return running_loss / max(len(loader), 1), macro_auroc


def save_checkpoint(model, path):
    torch.save({
        "model_state_dict": model.state_dict(),
        "temperature": model.temp_scaler.temperature.item(),
    }, path)

In [ ]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
history = []  # (epoch_global, phase, train_loss, val_loss, val_auroc)
best_val_auroc = -1.0
epoch_global = 0
best_ckpt_path = CKPT_DIR / "best_model.pth"
t_start, last_epoch_s, stopped_by_time = time.time(), 0.0, False


def run_phase(phase, n_epochs, optimizer, scheduler=None):
    global epoch_global, best_val_auroc, last_epoch_s, stopped_by_time
    for _ in range(n_epochs):
        # stop if one more epoch (+20 % margin) would pass the time budget
        if stopped_by_time or (time.time() - t_start + 1.2 * last_epoch_s) / 3600 > TRAIN_BUDGET_H:
            stopped_by_time = True
            print(f"  [time budget] stopping before epoch {epoch_global + 1} ({TRAIN_BUDGET_H} h limit)")
            return
        epoch_global += 1
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_auroc = validate(model, val_loader, criterion, device)
        if scheduler is not None:
            scheduler.step(val_loss)
        last_epoch_s = time.time() - t0
        lr = optimizer.param_groups[0]["lr"]
        print(f"  Epoch {epoch_global:>3} [{phase}]  train: {train_loss:.4f}  val: {val_loss:.4f}  "
              f"val_auroc: {val_auroc:.4f}  lr: {lr:.1e}  ({last_epoch_s:.0f}s)")
        history.append((epoch_global, phase, train_loss, val_loss, val_auroc))
        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            save_checkpoint(model, best_ckpt_path)
            print(f"    new best val macro AUROC: {best_val_auroc:.4f}")


# ── Phase 1: head warm-up (backbone frozen) ─────────────────────────────────
print("=" * 60, "\nPHASE 1 — Head warm-up (backbone frozen)\n", "=" * 60, sep="")
freeze_backbone(model)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_WARMUP, weight_decay=1e-4)
run_phase("warmup", EPOCHS_WARMUP, optimizer)

# ── Phase 2: full fine-tune (backbone unfrozen) ─────────────────────────────
print("\n" + "=" * 60, "\nPHASE 2 — Full fine-tune (backbone unfrozen)\n", "=" * 60, sep="")
unfreeze_backbone(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
run_phase("finetune", EPOCHS_FINETUNE, optimizer, scheduler)

print(f"\nTraining complete in {(time.time() - t_start) / 3600:.2f} h. "
      f"Best val macro AUROC: {best_val_auroc:.4f}  ({best_ckpt_path})")

In [ ]:
hist_df = pd.DataFrame(history, columns=["epoch", "phase", "train_loss", "val_loss", "val_auroc"])
warmup_end = (hist_df["phase"] == "warmup").sum()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(hist_df["epoch"], hist_df["train_loss"], color=C_BLUE, linewidth=2, label="Train loss")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], color=C_ORANGE, linewidth=2, label="Val loss")
axes[0].axvline(warmup_end + 0.5, color="#c3c2b7", linestyle="--", linewidth=1)
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE loss"); axes[0].set_title("Loss")
axes[0].legend(frameon=False); axes[0].spines[["top", "right"]].set_visible(False)
axes[0].grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0); axes[0].set_axisbelow(True)

axes[1].plot(hist_df["epoch"], hist_df["val_auroc"], color=C_AQUA, linewidth=2, label="Val macro AUROC")
axes[1].axvline(warmup_end + 0.5, color="#c3c2b7", linestyle="--", linewidth=1)
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro AUROC"); axes[1].set_title("Val AUROC (checkpoint metric)")
axes[1].legend(frameon=False); axes[1].spines[["top", "right"]].set_visible(False)
axes[1].grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0); axes[1].set_axisbelow(True)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "training_curves.png", dpi=140)
plt.show()


## 7. Post-hoc Temperature Calibration

Reload the best checkpoint, freeze everything except the single temperature
parameter, and tune it on the validation set (minimizing BCE) so that
`sigmoid(logits / T)` gives better-calibrated probabilities than the raw
logits — useful downstream if you later add a confidence-based routing rule.


In [ ]:
def tune_temperature(model, loader, device, lr=0.01, max_iter=100):
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    model.temp_scaler.log_temperature.requires_grad = True

    all_logits, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                all_logits.append(model.forward_logits(images.to(device)).float())
            all_labels.append(labels.to(device))
    all_logits = torch.cat(all_logits)
    all_labels = torch.cat(all_labels)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.LBFGS([model.temp_scaler.log_temperature], lr=lr, max_iter=max_iter)

    def _closure():
        optimizer.zero_grad()
        loss = criterion(model.temp_scaler(all_logits), all_labels)
        loss.backward()
        return loss

    optimizer.step(_closure)
    for p in model.parameters():
        p.requires_grad = True
    return model.temp_scaler.temperature.item()


ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

final_T = tune_temperature(model, val_loader, device)
print(f"Calibrated temperature: {final_T:.4f}")

calibrated_ckpt_path = CKPT_DIR / "best_model_calibrated.pth"
save_checkpoint(model, calibrated_ckpt_path)
print(f"Saved calibrated checkpoint: {calibrated_ckpt_path}")

## 8. Evaluation on the Official Test Set (`test_list.txt`)

Raw logits (before temperature) are computed once for val and test. They are reused for threshold
tuning, the test report, and the export in section 9. Nothing in this section is chosen using test data.

In [ ]:
@torch.no_grad()
def predict_logits(model, loader, device):
    """Raw logits (before temperature) and labels, in loader order (= dataframe order)."""
    model.eval()
    all_logits, all_labels = [], []
    for images, labels in loader:
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            logits = model.forward_logits(images.to(device, non_blocking=True))
        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.numpy())
    return np.concatenate(all_logits), np.concatenate(all_labels)


def to_probs(logits):
    return torch.sigmoid(torch.from_numpy(logits) / final_T).numpy()


val_logits, val_labels = predict_logits(model, val_loader, device)
test_logits, test_labels = predict_logits(model, test_loader, device)
val_probs, test_probs = to_probs(val_logits), to_probs(test_logits)
test_preds = (test_probs >= 0.5).astype(int)

# Per-class AUROC — skip classes with < 2 positive (or < 2 negative) examples
# in the test split, since AUROC is undefined for a single-class target.
auroc_per_class = {}
for i, label in enumerate(DISEASE_LABELS):
    y_true = test_labels[:, i]
    if len(np.unique(y_true)) < 2:
        auroc_per_class[label] = np.nan
        continue
    auroc_per_class[label] = roc_auc_score(y_true, test_probs[:, i])

auroc_series = pd.Series(auroc_per_class).sort_values(ascending=False)
print("Per-class AUROC (official test set):\n")
print(auroc_series.round(3).to_string())

macro_auroc = auroc_series.dropna().mean()
print(f"\nMacro-average AUROC (official test set, {len(test_df):,} images): {macro_auroc:.4f}")

print("\nClassification report @ threshold 0.5:\n")
print(classification_report(test_labels, test_preds, target_names=DISEASE_LABELS,
                             zero_division=0))

### 8b. Per-Class Threshold Tuning

The 0.5 cutoff above is the same for all 14 diseases, but each class's predicted-probability distribution sits in a different range (driven partly by each class's `pos_weight`). Tuning a separate threshold per class — chosen on the **validation** set, applied to the **test** set — fixes the low precision without needing to retrain anything.


In [ ]:
from sklearn.metrics import f1_score

best_thresholds = np.full(NUM_CLASSES, 0.5)
for i in range(NUM_CLASSES):
    y_true = val_labels[:, i]
    if len(np.unique(y_true)) < 2:
        continue
    candidates = np.arange(0.05, 0.95, 0.02)
    scores = [f1_score(y_true, (val_probs[:, i] >= t).astype(int), zero_division=0)
              for t in candidates]
    best_thresholds[i] = candidates[int(np.argmax(scores))]

print("Tuned per-class thresholds (val set):\n")
for l, t in zip(DISEASE_LABELS, best_thresholds):
    print(f"  {l:<20s} {t:.2f}")

# Re-score the TEST set with tuned thresholds instead of a flat 0.5
test_preds_tuned = (test_probs >= best_thresholds[None, :]).astype(int)
print("\nClassification report — tuned thresholds (test set):\n")
print(classification_report(test_labels, test_preds_tuned, target_names=DISEASE_LABELS,
                             zero_division=0))

from sklearn.metrics import hamming_loss, accuracy_score

# Per-class accuracy (tuned thresholds)
per_class_acc = (test_preds_tuned == test_labels).mean(axis=0)
acc_series = pd.Series(per_class_acc, index=DISEASE_LABELS).sort_values(ascending=False)
print("Per-class accuracy (tuned thresholds):\n")
print(acc_series.round(3).to_string())

# Two dataset-level accuracy numbers — they answer different questions:
hamming_acc = 1 - hamming_loss(test_labels, test_preds_tuned)      # avg correctness per label
subset_acc = accuracy_score(test_labels, test_preds_tuned)         # ALL 14 labels correct at once

print(f"\nHamming accuracy (avg per-label correctness): {hamming_acc:.3f}")
print(f"Subset accuracy (all 14 labels exactly right):  {subset_acc:.3f}")
print("\n[NOTE] Hamming accuracy looks high mostly because most labels are")
print("       negative for most images \u2014 a model that predicts 'no disease'")
print("       everywhere would already score ~90%+ here. Macro AUROC and")
print("       per-class F1 above are what actually measure whether the model")
print("       is finding real disease cases, not just being right by default.")

In [ ]:
plot_series = auroc_series.dropna().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(plot_series.index, plot_series.values, color=C_BLUE, height=0.65)
ax.axvline(0.5, color="#c3c2b7", linestyle="--", linewidth=1)
ax.text(0.505, -0.6, "random (0.5)", fontsize=8, color="#898781")
ax.set_xlim(0, 1)
ax.set_xlabel("AUROC")
ax.set_title("Per-class AUROC — official test set")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e1e0d9", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "auroc_per_class.png", dpi=140)
plt.show()

In [ ]:
# A handful of test predictions vs. ground truth
sample_idx = np.random.default_rng(SEED).choice(len(test_df), size=min(8, len(test_df)), replace=False)

fig, axes = plt.subplots(2, 4, figsize=(15, 7.5))
for ax, idx in zip(axes.flat, sample_idx):
    row = test_df.iloc[idx]
    img = Image.open(image_index[row["Image Index"]]).convert("L")
    ax.imshow(img, cmap="gray")

    true_labels = [l for l in DISEASE_LABELS if row[l] == 1] or ["No Finding"]
    pred_labels = [DISEASE_LABELS[i] for i in range(NUM_CLASSES) if test_preds[idx, i] == 1] or ["No Finding"]
    top_i = int(np.argmax(test_probs[idx]))
    title = (f"true: {', '.join(true_labels)}\n"
             f"pred: {', '.join(pred_labels)}\n"
             f"top p({DISEASE_LABELS[top_i]})={test_probs[idx, top_i]:.2f}")
    ax.set_title(title, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "test_predictions.png", dpi=140)
plt.show()


## 9. Save Outputs

* `results.json` — summary metrics, split id + hash, epochs actually run.
* `NIH_CXR14__NIH_OFFICIAL_PV__<MODEL_ID>__s42/` — `logits_val.npz`, `logits_test.npz` (raw logits,
  same format as the cascade runs), `train_log.csv`, `config.json`. Use these to compare this model
  with M08/M09 on the same images (e.g. with the patient-level bootstrap from the cascade notebook).

In [ ]:
epochs_run = {"warmup": int((hist_df["phase"] == "warmup").sum()),
              "finetune": int((hist_df["phase"] == "finetune").sum())}
results = {
    "model_id": MODEL_ID,
    "backbone": BACKBONE,
    "debug": DEBUG,
    "split_id": SPLIT_ID,
    "split_sha256": SPLIT_SHA256,
    "seed": SEED,
    "num_classes": NUM_CLASSES,
    "dataset_size": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "epochs_planned": {"warmup": EPOCHS_WARMUP, "finetune": EPOCHS_FINETUNE},
    "epochs_run": epochs_run,
    "stopped_by_time_budget": stopped_by_time,
    "best_val_macro_auroc": float(best_val_auroc),
    "calibrated_temperature": float(final_T),
    "macro_auroc_test": float(macro_auroc),
    "per_class_auroc_test": {k: (None if np.isnan(v) else float(v)) for k, v in auroc_per_class.items()},
    "tuned_thresholds": {l: float(t) for l, t in zip(DISEASE_LABELS, best_thresholds)},
    "per_class_accuracy_test": {l: float(a) for l, a in zip(DISEASE_LABELS, per_class_acc)},
    "hamming_accuracy_test": float(hamming_acc),
    "subset_accuracy_test": float(subset_acc),
}

with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(results, f, indent=2)

# ── Export raw logits in the cascade-study format (evaluation_protocol.yaml: outputs.per_run_folder) ──
RUN_DIR.mkdir(parents=True, exist_ok=True)
for part, frame, z, y in (("val", val_df, val_logits, val_labels), ("test", test_df, test_logits, test_labels)):
    y_frame = frame[DISEASE_LABELS].to_numpy(dtype=np.uint8)
    assert np.array_equal(y.astype(np.uint8), y_frame), f"{part}: loader order does not match the dataframe"
    np.savez(RUN_DIR / f"logits_{part}.npz", image_index=frame["Image Index"].to_numpy(dtype=str),
             patient_id=frame["Patient ID"].to_numpy(dtype=np.int64), logits=z.astype(np.float32),
             labels=y_frame, label_order=json.dumps(DISEASE_LABELS),
             split_id=SPLIT_ID, split_sha256=SPLIT_SHA256, model_id=MODEL_ID, seed=SEED, eval_set=part)
hist_df.to_csv(RUN_DIR / "train_log.csv", index=False)
with open(RUN_DIR / "config.json", "w") as f:
    json.dump({"model_id": MODEL_ID, "backbone": BACKBONE, "seed": SEED, "split_id": SPLIT_ID,
               "split_sha256": SPLIT_SHA256, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE,
               "lr_warmup": LR_WARMUP, "lr_finetune": LR_FINETUNE, "epochs_run": epochs_run,
               "calibrated_temperature": float(final_T), "torch": torch.__version__}, f, indent=2)

print(json.dumps(results, indent=2))
print(f"\nCheckpoints : {CKPT_DIR}")
print(f"Figures     : {FIGURES_DIR}")
print(f"Results     : {OUTPUT_DIR / 'results.json'}")
print(f"Logits      : {RUN_DIR}")